# The subthreshold regime: a graded response

**This notebook is outside the scope of the paper.** Section III-C shows that when the
slope of the sigmoid stays below one, the resting point is globally asymptotically stable
for every constant input and for either selector, so no bifurcation is available and the
cell can never fire repetitively. That regime is a candidate for Hodgkin type III excitability
and the paper leaves it as future work. What follows is one minimal example of what the
loop does there.

The model is the band-pass loop of Section V run below criticality, $k < 1$, with a second
input port at the filter node:

$$\dot x_1 = x_2, \qquad
\dot x_2 = -\omega_n^2 x_1 - \tfrac{\omega_n}{Q} x_2 + y + u_c, \qquad
y = \tanh\!\big(k (u_v + w)\big), \qquad w = \tfrac{\omega_n}{Q} x_2 .$$

The filter port is the analogue of an applied current in a conductance-based model. At any
constant input, $x_2 = 0$ and $w = 0$, so $u_c$ shifts $x_1$ and nothing else: it drives
transients but cannot modulate excitability, which is what the sigmoid port does.

In [ ]:
using Plots, LaTeXStrings, DifferentialEquations, Printf, ProgressMeter, Plots.PlotMeasures

gr(guidefontsize = 14, tickfontsize = 12, legendfontsize = 12, margin = 5Plots.mm, grid = true)
myApple  = RGBA(187/255, 206/255, 131/255, 1)
myBlue   = RGBA(131/255, 174/255, 218/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPurple = RGBA(169/255,  90/255, 179/255, 1)
default(fmt = :png);

In [ ]:
Base.@kwdef struct T3Neuron
    k::Float64   = 0.7        # sigmoid gain, k < 1 leaves the loop globally stable
    Q::Float64   = 0.5        # quality factor
    fc::Float64  = 1.0        # center frequency [Hz]
    uv::Function = t -> 0.0   # sigmoid port
    uc::Function = t -> 0.0   # filter port
end

omega(p::T3Neuron) = 2pi * p.fc

step_input(a, t0) = t -> (t < t0 ? 0.0 : a)

function rhs!(dX, X, p::T3Neuron, t)
    x1, x2 = X
    y = tanh(p.k * (p.uv(t) + (omega(p) / p.Q) * x2))
    dX[1] = x2
    dX[2] = -omega(p)^2 * x1 - (omega(p) / p.Q) * x2 + y + p.uc(t)
    return nothing
end

# resting point for constant inputs: x2 = 0, w = 0, y = tanh(k*uv)
equilibrium(p::T3Neuron, t = 0.0) = [(tanh(p.k * p.uv(t)) + p.uc(t)) / omega(p)^2, 0.0]

function simulate(p::T3Neuron; tspan = (0.0, 12.0), tstops = Float64[], saveat = 0.002)
    solve(ODEProblem(rhs!, equilibrium(p, tspan[1]), tspan, p), Tsit5();
          abstol = 1e-10, reltol = 1e-9, saveat = saveat, tstops = tstops)
end

function signals(sol, p::T3Neuron)
    t  = sol.t
    w  = (omega(p) / p.Q) .* [s[2] for s in sol.u]
    uv = p.uv.(t)
    return (t = t, w = w, uv = uv, uc = p.uc.(t), y = tanh.(p.k .* (uv .+ w)))
end

peak_y(s; t0 = 0.0) = maximum(s.y[findfirst(x -> x >= t0, s.t):end])

## One transient, then silence

A step at the filter port produces a single excursion at the onset and nothing afterwards.
The constant part of the step is invisible once the transient is over, since the selector
blocks it. The cell reports the derivative of the input rather than its level.

In [ ]:
p = T3Neuron(k = 0.7, Q = 0.5, fc = 1.0, uc = step_input(2.0, 2.0))
s = signals(simulate(p; tspan = (0.0, 12.0), tstops = [2.0]), p)

@printf("peak y after the step = %.4f,   |y| at t = 8 s = %.2e\n",
        peak_y(s; t0 = 2.0), abs(s.y[findfirst(x -> x >= 8.0, s.t)]))

p1 = plot(s.t, s.uc; lw = 2, c = :black,   ylabel = L"u_c", legend = false)
p2 = plot(s.t, s.w;  lw = 2, c = myBlue,   ylabel = L"w",   legend = false)
p3 = plot(s.t, s.y;  lw = 2, c = myPurple, ylabel = L"y",   xlabel = "time [s]",
          ylims = (-1.05, 1.05), legend = false)
plot(p1, p2, p3; layout = (3, 1), size = (900, 560), link = :x,
     title = ["k = $(p.k),  Q = $(p.Q)" "" ""], titlefontsize = 11)

## The response is graded

Sweeping the amplitude of the step over three decades gives a curve with no jump anywhere.
Nothing here is all-or-none: the peak grows smoothly with the stimulus and the only
compression comes from the sigmoid itself, since a globally stable loop adds no
sharpening. This is the contrast with the two excitable regimes of the paper, where the
onset is a fold or a Hopf and the spike has full amplitude as soon as it exists.

In [ ]:
As   = exp.(range(log(0.02), log(60); length = 40))
ks   = [0.3, 0.5, 0.7, 0.9]
cols = [myBlue, myApple, myPurple, myOrange]

fig = plot(xlabel = "step amplitude at the filter port", ylabel = "peak y",
           xscale = :log10, size = (900, 500), legend = :topleft,
           title = "graded response, no threshold, Q = 0.5", titlefontsize = 14)
@showprogress for (k, c) in zip(ks, cols)
    pk = Float64[]
    for A in As
        q = T3Neuron(k = k, Q = 0.5, fc = 1.0, uc = step_input(A, 2.0))
        push!(pk, peak_y(signals(simulate(q; tspan = (0.0, 12.0), tstops = [2.0]), q); t0 = 2.0))
    end
    plot!(fig, As, pk; lw = 3, c = c, label = "k = $k")
end
ylims!(fig, 0, 1.05)
fig